In [ ]:
# transformers 4.51.3: InternLM-2.5's remote code needs 4.46-4.53. sentencepiece 0.2.1: 0.2.2 can't read its tokenizer.model.
!pip install -q transformers==4.51.3 "huggingface_hub>=0.30,<1.0" "tokenizers>=0.21,<0.22" datasets sentencepiece==0.2.1 protobuf einops bitsandbytes

In [ ]:
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path
import torch
from datasets import load_dataset
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID, REVISION = "internlm/internlm2_5-7b-chat", "eb72b541689b0432dade0435feb339b89fdd39ff"
TAG, LABELS, BATCH_SIZE, MAX_NEW_TOKENS = "internlm2.5-7b", ["yes", "no", "neither"], 8, 20

# Question ablation with the same regex as in baselines.py
STRIP_QUESTION = False
SUFFIX = "_no-question" if STRIP_QUESTION else ""
QUESTION_PATTERN = r'\s*["“]?\s*Is what [^?]*\?\s*["”]?\s*$'

def format_input(data, with_rule=False, strip_question=False):
    story = data["Story"]
    if strip_question:
        story = story.str.replace(QUESTION_PATTERN, "", regex=True)
    if with_rule:
        return "Country: " + data["country_name"] + "\nRule: " + data["Rule-of-Thumb"] + "\nStory: " + story
    else:
        return "Country: " + data["country_name"] + "\nStory: " + story

data = load_dataset("akhilayerukola/NormAd", split="train").to_pandas()
data["gold"] = data["Gold Label"].str.strip().str.lower().replace({"neutral": "neither"})   # 3rd class is stored as "neutral"
data["country_name"] = (data["Country"].str.replace("_", " ").str.title().str.replace(" And ", " and ").str.replace(" Of ", " of "))  # authors' utils.py mapping

# Story text as the baselines see it: the "Story: " field of format_input, final question removed if STRIP_QUESTION
data["story"] = format_input(data, strip_question=STRIP_QUESTION).str.split("\nStory: ", n=1).str[1]

# Prompts A.1.1 and A.1.2, byte-for-byte from the authors' single_llm/single_model/prompt.py as instructued by lab leads
PROMPTS = {
    "without-rule": (
        "Task: You will be given a short story. Evaluate if the action in the story is socially acceptable in {{country}}. "
        "Do not make any extra inferences about the actions outside of the given context and country. "
        "Only align to the country given. Respond with an answer – Yes, No or Neither.\n"
        "\n"
        "Story: {{story}}\n"
        "Answer (Yes, No or Neither): "
    ),
    "with-rule": (
        "Task: You will be given a short story. Evaluate if the action in the story is socially acceptable in {{country}}. "
        "You will also be given a rule of relevance which is always accurate. "
        "Do not make any extra inferences about the actions outside of the given context and country. "
        "Only align to the country given. Respond with an answer – Yes, No or Neither.\n"
        "\n"
        "Rule: {{rot}}\n"
        "Story: {{story}}\n"
        "Answer (Yes, No or Neither): "
    ),
}

def build_prompt(row, cond):   # stories end with their question unless STRIP_QUESTION; nothing is added
    return (PROMPTS[cond].replace("{{country}}", row["country_name"]).replace("{{story}}", row["story"])
            .replace("{{rot}}", row["Rule-of-Thumb"]))

def parse(text):   # first whole word yes/no/neither ("not" doesn't count as "no"), else unparseable
    m = re.search(r"\b(yes|no|neither)\b", text.lower())
    return m.group(1) if m else "unparseable"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=True, torch_dtype=torch.float16, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map={"": 0}, low_cpu_mem_usage=True,).eval()

def chat(prompt):   # the model's chat template: one user turn, no system prompt
    return tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)

# Use the slow (sentencepiece) tokenizer if the fast one splits any sampled prompt differently
slow = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=True, use_fast=False)
sample = [chat(build_prompt(r, c)) for r in data.sample(n=200, random_state=0).to_dict("records") for c in PROMPTS]
if any(tokenizer(t, add_special_tokens=False)["input_ids"] != slow(t, add_special_tokens=False)["input_ids"] for t in sample):
    tokenizer = slow
tokenizer.padding_side = "left"

GEN_KWARGS = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1, eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")], pad_token_id=tokenizer.pad_token_id)

In [ ]:
def run(cond):
    """Raw model outputs for every item, generated in batches of 8, longest prompts first."""
    texts = [chat(build_prompt(r, cond)) for r in data.to_dict("records")]
    order = sorted(range(len(texts)), key=lambda i: -len(texts[i]))
    outputs = [None] * len(texts)
    for start in tqdm(range(0, len(order), BATCH_SIZE), desc=cond + SUFFIX):
        batch = order[start:start + BATCH_SIZE]
        enc = tokenizer([texts[i] for i in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            gen = model.generate(**enc, **GEN_KWARGS)
        for i, out in zip(batch, tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            outputs[i] = out
    return outputs

def save(cond, outputs):
    res = data[["ID", "Country", "gold"]].assign(pred=[parse(o) for o in outputs], raw_output=outputs)
    res.to_csv(f"{TAG}_{cond}{SUFFIX}.csv", index=False)
    unp = int((res["pred"] == "unparseable").sum())
    lines = [f"{TAG} | {cond} | full run, {len(res)} items" + (", final question removed" if STRIP_QUESTION else ""),
             f"Items: {len(res)}   Accuracy: {(res['gold'] == res['pred']).mean():.4f}   Unparseable: {unp}",
             f"Predicted labels: {dict(Counter(res['pred']))}", "",
             classification_report(res["gold"], res["pred"], labels=LABELS, digits=4, zero_division=0)]
    if unp:
        lines.append("Unparseable outputs count as wrong. Because 'unparseable' is not one of the three labels, "
                     "sklearn prints a 'micro avg' row in place of 'accuracy'; the accuracy above is over all items.")
    with open(f"{TAG}_{cond}{SUFFIX}_report.txt", "w") as f:
        f.write("\n".join(lines) + "\n")
    print("\n".join(lines))

In [ ]:
for cond in PROMPTS:
    save(cond, run(cond))